#### Address Github Compatibility for nbformat

In [4]:
import json

with open('LLM_Medical_Assistant_Prompt_Engineering.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

nb.get('metadata', {}).pop('widgets', None)

with open('LLM_Medical_Assistant_Prompt_Engineering.ipynb', 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=1)

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

To develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --no-cache-dir --no-deps -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 89.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
# For installing the libraries & downloading models from HF Hub
!pip install --upgrade huggingface_hub==0.35.3 pandas==2.2.2 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 sentence-transformers==5.1.1  -q

In [4]:
pip install diskcache llama-cpp-python==0.2.28 --no-deps --no-cache-dir -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 175.9 MB/s eta 0:00:00


In [ ]:
!pip install faiss-gpu-cu11

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 45.7 MB/s eta 0:00:00


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [14]:
#Libraries for processing dataframes,text
import json,os
import pandas as pd
import numpy as np

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [6]:
import textwrap
import warnings
warnings.filterwarnings('ignore')

#### function to pretty print a collection

In [7]:
def pretty_print_doc_collection(relevant_document_chunks):
  for i, chunk in enumerate(relevant_document_chunks):
    print(f"── Chunk {i+1} ──────────────────────────────────────────")
    print(f"Page   : {chunk['metadata'].get('page', 'N/A')}")
    print(f"Score  : {chunk['score']:.4f}")
    print(f"Text   :")
    print(textwrap.fill(chunk['text'], width=80))
    print()

## Basic LLM Response evalution function

In [8]:
def evaluate_response(response):
    return {
        "word_count":        len(response.split()),
        "has_disclaimer":    any(w in response.lower() for w in ["consult", "disclaimer"]),
        "is_structured":     any(c in response for c in ["1.", "•", "-", "\n"]),
        "mentions_treatment": any(w in response.lower() for w in ["intervention", "management", "treatment","therapy","medication","dose"]),
        "mentions_symptom":   any(w in response.lower() for w in ["recognition", "symptom","sign","present","diagnos"]),
    }

# Question Answering using LLM

Following Google Colab T4 friendly LLMs on huggingface were analyzed and compared:

| Model | Params | Domain | Strengths | Weaknesses | Colab T4 Friendly | Notes |
|------|------|------|------|------|------|------|
| meta-llama/Meta-Llama-3-8B-Instruct | 8B | General | Excellent reasoning, strong benchmarks, good instruction following | Not medical-specific | Runs with 4-bit quantization | Best overall |
| mistralai/Mistral-7B-Instruct-v0.2 | 7B | General | Fast inference, strong reasoning, widely used in RAG systems | Slightly weaker knowledge depth | Runs easily on T4 | Best for speed |
| epfl-llm/meditron-7b | 7B | Medical | Trained on PubMed and clinical texts | Weaker instruction following | Runs on T4 | Good domain baseline |
| BioMistral-7B | 7B | Medical | Biomedical pretraining improves performance over MediTron on some tasks | Some hallucination issues | Runs on T4 | Good medical candidate |
| OpenBioLLM-8B | 8B | Medical | Llama-3 based medical tuning | Less widely tested | Runs on T4 with quantization | Promising experimental |

Purpose is to use the LLM model to act as a medical assistant answering natural language questions. Based on Strengths and Weeknesses in the table, **meta-llama/Meta-Llama-3-8B-Instruct** is selected for response generation for its strong ability to generate:

* Structured explanations

* Follow prompts correctly

* Produce step-by-step reasoning


#### **Downloading the model from Hugging Face**

In [9]:
model_name_or_path = "bartowski/Meta-Llama-3-8B-Instruct-GGUF"
model_basename = "Meta-Llama-3-8B-Instruct-Q4_K_M.gguf" # the model is in gguf format

In [10]:
from huggingface_hub import login
login()

In [15]:
bartowski_model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

Meta-Llama-3-8B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

#### **Model Configuration**
* Max context of 5000 tokens (Max allowed 8192 Tokens)
* Llama 3 8B has 32 transformer layers so 38 layers means every layer runs on GPU
* Process 512 tokens batch in parallel

In [16]:
#Runtime is connected to GPU.
llm = Llama(
    model_path=bartowski_model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### **Model Parameters**

I have set model parameters for **deterministic** response.

* temperature=0 - always pick highest probability token
* top_p=0.95 - probability mass cutoff
* top_k=10 means: only consider top 10 tokens
* Max output tokens 256, Response gets truncated beyond that


#### Function to generate response

In [17]:
def llm_response(query,max_tokens=256,temperature=0,top_p=0.95,top_k=10):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

#### **Declare Query Constants**

In [18]:
from typing import Final
Query1: Final[str] = "What is the protocol for managing sepsis in a critical care unit?"
Query2: Final[str] = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
Query3: Final[str] = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
Query4: Final[str] = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
Query5: Final[str] = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [19]:
response = llm_response(Query1)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

 The protocol should include steps for identifying and treating patients with sepsis, as well as strategies for preventing sepsis.
The protocol for managing sepsis in a critical care unit typically includes the following steps:
1. Identification of Sepsis: Sepsis is identified by the presence of two or more of the following criteria:
* Temperature > 38°C (100.4°F) or < 36°C (96.8°F)
* Heart rate > 90 beats per minute
* Respiratory rate > 20 breaths per minute
* White blood cell count > 12,000 cells/mm³ or < 400 cells/mm³
* Sepsis is also suspected if a patient has a known infection and develops organ dysfunction.
2. Initial Assessment: Upon identification of sepsis, the critical care team should perform an initial assessment to determine the severity of illness and identify potential sources of infection.
3. Fluid Resuscitation: Patients with sepsis should receive fluid resuscitation to maintain adequate blood pressure and perfusion. This may involve administering crystalloid or colloi

{'word_count': 180,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
response = llm_response(Query2)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

Llama.generate: prefix-match hit


 Appendicitis is a medical condition that occurs when the appendix becomes inflamed and fills with pus. The appendix is a small, finger-like pouch attached to the large intestine.
Common Symptoms of Appendicitis:
1. Severe abdominal pain: The most common symptom of appendicitis is severe abdominal pain that starts near the belly button and then moves to the lower right side of the abdomen.
2. Nausea and vomiting: Many people with appendicitis experience nausea and vomiting, which can be accompanied by fever, chills, and loss of appetite.
3. Abdominal tenderness: The abdomen may become tender to the touch, especially in the lower right quadrant.
4. Fever: A high fever is common in people with appendicitis, often above 100.4°F (38°C).
5. Loss of appetite: People with appendicitis may experience a loss of appetite and feel weak or fatigued.

Can Appendicitis be Cured via Medicine?
Appendicitis cannot be cured solely through medicine. Antibiotics may help alleviate symptoms and reduce the 

{'word_count': 187,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': True}

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
response = llm_response(Query3)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

Llama.generate: prefix-match hit


 Hair loss can be a distressing experience, especially when it occurs suddenly and unexpectedly. In this article, we will explore some of the most common causes and effective treatments for sudden patchy hair loss.
Causes of Sudden Patchy Hair Loss:
1. Alopecia Areata: This is an autoimmune condition where the immune system attacks healthy hair follicles, leading to patchy hair loss.
2. Telogen Effluvium: This is a condition where there is an excessive shedding of hair due to hormonal changes, stress, or nutritional deficiencies.
3. Traction Alopecia: This occurs when hair is pulled too tightly, causing hair loss at the scalp.
4. Fungal Infections: Fungal infections like ringworm can cause patchy hair loss.
5. Nutritional Deficiencies: Lack of essential nutrients like iron, zinc, and biotin can contribute to hair loss.
6. Hormonal Imbalance: Hormonal changes during pregnancy, menopause, or thyroid disorders can lead to hair loss.
7. Stress: Physical or emotional stress can cause hair l

{'word_count': 180,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
response = llm_response(Query4)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

Llama.generate: prefix-match hit


 The answer depends on the severity and location of the injury, as well as the individual's overall health. Here are some common treatment options:
1. Rest: In cases of mild traumatic brain injury (mTBI), rest is often recommended to allow the brain time to heal.
2. Medications: Pain relievers, anti-anxiety medications, and sleep aids may be prescribed to manage symptoms such as headaches, anxiety, or insomnia.
3. Physical therapy: Rehabilitation programs can help improve strength, balance, coordination, and cognitive function.
4. Occupational therapy: This type of therapy focuses on helping individuals with brain injuries regain daily living skills, such as dressing, grooming, and cooking.
5. Speech therapy: Speech therapists can help individuals with language processing difficulties or communication problems.
6. Cognitive rehabilitation: This type of therapy aims to improve memory, attention, problem-solving, and other cognitive functions.
7. Neurorehabilitation: This comprehensive a

{'word_count': 195,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
response = llm_response(Query5)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

 A fractured leg can be a serious injury that requires immediate attention and proper care to ensure optimal healing and minimize complications. Here's a comprehensive guide on how to handle such an emergency situation:

**Initial Response**

1. **Stop the bleeding**: Apply direct pressure to the wound using a clean cloth or gauze for at least 10-15 minutes to control any bleeding.
2. **Immobilize the leg**: Use a splint, sling, or crutches to immobilize the affected leg and reduce pain and swelling.
3. **Assess the injury**: Check for signs of shock, such as pale or cool skin, rapid pulse, and decreased blood pressure.

**Emergency Medical Treatment**

1. **Call 911 or local emergency services**: If you're in a remote area, call for medical help immediately.
2. **Transport to a hospital**: Get the injured person to a hospital or medical facility as quickly and safely as possible.
3. **Provide basic first aid**: Continue to apply direct pressure to any bleeding wounds and maintain immo

{'word_count': 190,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### **Comments and Observations**

Following observations were made from the answers received from "Meta-Llama-3-8B-Instruct-GGUF"

* Without external knowledge source (RAG) the model relies on general training data. It does not have access Merck Manual and may use outdated medical material that it was trained on. E.g. The model seem to use outdated criterian for Sepsis identification.

* LLM does not provide any citation of its source of information.

* On Appendicitis query, model missed clinical signs, diagnostics, and treatment details. For query 3 ( patchy hair loss) model did reasonably good job in answering althought specifics for clinical diagnosis and treatment were missing.

* In query 5, LLM assumed there is bleeding but the query only mentioned fracture. Model changes the response structure, instead of earlier responses where bullet point response was given, query 5 returns response in sections.

This behavior is likley because model is anwering from training on various public documents, like Wikipedia, medical websites, health articles, research papers. Since model is providing answers that are not clinically accurate and complete, this approach can't be used as such.

Next step is to try to improve model response by using **prompt engineering**.

# Question Answering using LLM with Prompt Engineering

The goal here is to use **prompt engineering** to guide the model to reason, structure, and avoid hallucinations without adding external source of knowledge.

Idea is to create expert instruction prompts that forces the model to assume a professional role, provide structured answers by separating causes, symptoms, treatments and for safety add clinical caution and avoid unsupported claims.

We will apply **Instruction Prompting** and **Output Structuring** to structure the content for the purpose of being useful for human consumption. This will be different than structuring for machine consumption.

We will divide the prompt into **System Prompt** and **User prompt**. System prompt will be used for **Role prompting**, **Structured output**, **Safety constraints**

In addition to prompt engineering, we will try various hyperparameter configurations for our LLM, trying settings ranging from **Deterministic behavior** to **Exploratory response**. The parameters for all 5 settings are given in below table:

| Strategy | temperature | top_p | top_k | max_tokens | Use Case |
|---|---|---|---|---|---|
| Deterministic | 0 | 0.95 | 10 | 256 | Factual, consistent answers |
| Conservative | 0.1 | 0.9 | 20 | 256 | Slight variation, still safe |
| Balanced | 0.3 | 0.85 | 40 | 512 | General medical Q&A |
| Creative | 0.7 | 0.9 | 50 | 512 | Differential diagnosis |
| Exploratory | 1.0 | 0.95 | 100 | 1024 | Brainstorming, research |


* With Conservative Hyperparameter combination we will add **Constraint Prompting** by specifically adding some rules.
* With **Creative** and **Exploratory** Hyperparameter combination, we will add **Chain-of-Thought (CoT)** prompting to our template

In [ ]:
engineered_system_prompt = """
You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response
- Cite section titles if available
"""

In [ ]:
engineered_prompt_template = f"""
<SYSTEM>
{engineered_system_prompt}
</SYSTEM>

<USER>
###query
</USER>

<MEDICAL-ASSISTANT>
"""

In [ ]:
def get_engineered_prompt (query,prompt_template):
  engineered_prompt = prompt_template.replace('###query', query)
  return engineered_prompt

## **1. Deterministic Hyperparameter Optimization**
LLM transformer outputs can be made deterministic by setting temperature to 0, enabling greedy decoding (choosing the highest probability token). This is same confguration as our last run without prompt engineering.


* temperature=0 - always pick highest probability token
* top_p=0.95 - probability mass cutoff
* top_k=10 means: only consider top 10 tokens
* Max output tokens 256, Response gets truncated beyond that

In [ ]:
def llm_deterministic_response(prompt,max_tokens=256,temperature=0,top_p=0.95,top_k=10):
    model_output = llm(
      prompt=prompt,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
prompt = get_engineered_prompt(Query1, engineered_prompt_template)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_deterministic_response(prompt)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

******** PROMPT *****************

<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>

<USER>
What is the protocol for managing sepsis in a critical care unit?
</USER>

<MEDICAL-ASSISTANT>

******** RESPONSE *****************
**Sepsis Management Protocol**

**Definition:** Sepsis is a l

{'word_count': 158,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations
if we compare this response with our first experiment without prompt engineering:

```
{'word_count': 160,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': False}
```

The response with prompt engineering is given in **sections** as requested by our **system prompt** and mentions both treatment and symptomps. That is a **big improvement**.

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
prompt = get_engineered_prompt(Query2, engineered_prompt_template)
print("******** PROMPT *****************")
print(prompt)
response = llm_deterministic_response(prompt)
print("******** RESPONSE *****************")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

******** PROMPT *****************

<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>

<USER>
What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
</USER>

<MEDICAL-ASSISTANT>

******** RESPONSE *****************
**Appendicitis: Symptom

{'word_count': 162,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations
if we compare this response with our first experiment where no prompt engineering was applied:

```
{'word_count': 187,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': True}
```

 The response with **prompt engineering** additionally mentions treatment. That is a ** improvement**.

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
prompt = get_engineered_prompt(Query3, engineered_prompt_template)
print("******** PROMPT *****************")
print(prompt)
response = llm_deterministic_response(prompt)
print("******** RESPONSE *****************")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

******** PROMPT *****************

<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>

<USER>
What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?
</USER>

<MEDICAL-ASSISTANT>



Llama.generate: prefix-match hit


******** RESPONSE *****************
**Sudden Patchy Hair Loss: Causes and Treatments**

**Causes:**
Patchy hair loss can occur due to various factors. Some common causes include:

• **Alopecia Areata**: An autoimmune disorder that causes the immune system to attack healthy hair follicles, leading to patchy hair loss.
• **Telogen Effluvium**: A condition where there is an excessive shedding of hair due to hormonal changes, stress, or nutritional deficiencies.
• **Fungal Infections**: Fungal infections like ringworm can cause patchy hair loss on the scalp.
• **Scalp Irritation**: Irritation from harsh chemicals, heat styling tools, or tight hairstyles can lead to patchy hair loss.

**Treatments:**
The most effective treatments for sudden patchy hair loss depend on the underlying cause. Some common treatment options include:

• **Topical Corticosteroids**: Creams or ointments containing corticosteroids can help reduce inflammation and promote hair growth.
• **Minoxidil**: A topical soluti

{'word_count': 168,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}

### Observations
if we compare this response with our first experiment where no prompt engineering was applied:

```
{'word_count': 180,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

 The response with **prompt engineering** mentions both symptomps. That is an ** improvement**.

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
prompt = get_engineered_prompt(Query4, engineered_prompt_template)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_deterministic_response(prompt)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

******** PROMPT *****************

<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>

<USER>
What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
</USER>

<MEDICAL-ASSISTANT>

******** RESPONSE *****************


Llama.generate: prefix-match hit


**Brain Injury Treatment**

A person who has sustained a physical injury to brain tissue may require immediate medical attention. The goal of treatment is to stabilize the patient and prevent further damage.

**Initial Assessment and Stabilization**

* Emergency responders will assess the patient's airway, breathing, and circulation (ABCs) to ensure they are stable.
* Patients with severe head injuries may be taken to an intensive care unit (ICU) for close monitoring.
* Imaging studies such as computed tomography (CT) or magnetic resonance imaging (MRI) scans may be ordered to evaluate the extent of brain damage.

**Medical Management**

* Pain management: Medications such as acetaminophen, ibuprofen, or opioids may be prescribed to manage pain and discomfort.
* Anti-seizure medications: Patients with a history of seizures or those who have suffered a severe head injury may be prescribed anti-seizure medications to prevent seizures.
* Blood pressure control: Patients with increased int

{'word_count': 188,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations

if we compare this response with our first experiment where no prompt engineering was applied:

```
{'word_count': 195,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}
```

 The response with **prompt engineering** provides structured response as asked in system prompt. Interestingly, LLM response without prompt engineering, also mentioned **Neurorehabilitation** and **Rehabilitation centers**. Since our prompt did not ask LLM about Rehabilitation as part of System Prompt, that category was not included by LLM.

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
prompt = get_engineered_prompt(Query5, engineered_prompt_template)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_deterministic_response(prompt)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

******** PROMPT *****************

<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>

<USER>
What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?
</USER>

<MEDICAL-ASSISTANT>

******** RESPONSE *****************


Llama.generate: prefix-match hit


**Fractured Leg: Precautions and Treatment Steps**

**Initial Assessment and Stabilization**

* Immediately stop any activity that may have caused the fracture.
* Apply a splint or immobilize the affected limb to prevent further injury.
* Monitor vital signs, including pulse, blood pressure, and oxygen saturation.

**Emergency Medical Care**

* If the person is experiencing severe bleeding, apply direct pressure to the wound using a clean cloth.
* If there are no signs of bleeding, but the person is in severe pain or has difficulty moving the affected limb, consider transporting them to an emergency medical facility for further evaluation and treatment.

**Treatment Protocol**

* Immobilize the fractured leg with a splint or cast to prevent further injury and promote healing.
* Administer pain management medication as prescribed by a healthcare provider.
* Monitor for signs of infection, such as redness, swelling, warmth, or pus around the wound.
* Consider administering antibiotics if

{'word_count': 205,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

if we compare this response with our first experiment where no prompt engineering was applied:


```
 {'word_count': 190,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

```

 The response with **prompt engineering** provided Care and Recovery section while LLM response without prompt engineering didn't cover this topic.


## **2. Conservative Hyperparamter Optimization**
A conservative hyperparameter configuration for LLM transformers focuses on stability, high precision, and avoiding over-fitting, particularly when fine-tuning. This approach often uses low learning rates, smaller batch sizes, and lower sampling temperatures.

* temperature=0.1 — Near Greedy
* top_k=20 — Slightly Wider Than Deterministic
* top_p=0.90 sits exactly between balanced and exploratory
* max_tokens=256 is kept small to force the LLM to be concise as directed in
**system prompt**

* With **Consevative Hyperparameter** combination we will add **Constraint Prompting** by specifically adding some rules.

In [ ]:
engineered_constraint_prompt_template = f"""
<SYSTEM>
{engineered_system_prompt}
</SYSTEM>
<CONSTRIANTS>
1. Answer in under 5 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
###query
</USER>
<MEDICAL-ASSISTANT>
"""

In [ ]:
def llm_conservative_response(query,max_tokens=256,temperature=0.1,top_p=0.90,top_k=20):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
prompt = get_engineered_prompt(Query1, engineered_constraint_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_conservative_response(prompt)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 5 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What is the protocol for managing sepsis in a critical care unit?
</USER>
<MEDICAL-ASSISTANT>

******** RESPONSE *****************


Llama.generate: prefix-match hit


**Sepsis Management Protocol**

Sepsis is a life-threatening condition that occurs when an infection triggers an overwhelming inflammatory response throughout the body. The goal of sepsis management is to identify and treat the underlying cause, stabilize vital signs, and prevent organ dysfunction.

**Causes:**
• Infection (bacterial, viral, or fungal)
• Immune system dysregulation

**Treatment Protocol:**

1. **Initial Assessment:** Monitor vital signs, perform a physical examination, and obtain blood cultures.
2. **Fluid Resuscitation:** Administer IV fluids to maintain adequate perfusion.
3. **Antimicrobial Therapy:** Initiate broad-spectrum antibiotics or antifungals based on suspected pathogen.
4. **Supportive Care:** Provide oxygen therapy, mechanical ventilation if necessary, and manage pain and discomfort.

**Key Considerations:**

• Early recognition and treatment are crucial
• Monitor for signs of organ dysfunction (renal, respiratory, cardiovascular)
• Adjust treatment plan 

{'word_count': 170,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations

if we compare this response with our first experiment where no prompt engineering was applied:

```
{'word_count': 183,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```
The response with **prompt engineering** provides symptoms and disclaimer.
The Conservative Hyperparamter Optimization gives model little more freedom (temperature=0.1,top_p=0.90) than **Determinitic** behavior. With **System Prompt** we added rule that answer be limited to 5 sentences. Clearlu LLM has responsed in a concise to the point response and ended with "Consult a doctor for personal advice."

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
prompt = get_engineered_prompt(Query2, engineered_constraint_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_conservative_response(prompt)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 5 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure

Llama.generate: prefix-match hit


**Appendicitis: Symptoms and Treatment**

Common symptoms of appendicitis include:

* Severe abdominal pain that starts near the belly button and moves to the lower right side
* Nausea and vomiting
* Loss of appetite
* Fever
* Abdominal tenderness

Appendicitis is typically caused by a blockage in the appendix, which can lead to inflammation and infection.

**Treatment:**

While antibiotics may be used to treat appendicitis, surgery is usually necessary to remove the inflamed appendix. The most common surgical procedure for appendicitis is an open appendectomy, where the surgeon makes a single incision in the abdomen to access the appendix. In some cases, laparoscopic surgery or robotic-assisted surgery may also be used.

**Consult a doctor for personal advice.**
</MEDICAL-ASSISTANT>assistant

Here's a revised version of the response that meets the constraints:

<SYSTEM>
...
</SYSTEM>

<CONSTRIANTS>
1. Answer in under 5 sentences
2. Cite the condition, cause, and treatment
3. End with:

{'word_count': 167,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations

For Query2, **prompt engineering** and configuration modified behaviour of LLM exactly same as for Query1. ### Observations

if we compare this response with our first experiment where no prompt engineering was applied:

```
{'word_count': 187,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': True}
```

The response with **prompt engineering** provides symptoms and disclaimer.  The response with **prompt engineering** provides symptoms and disclaimer.
The Conservative Hyperparamter Optimization gives model little more freedom (temperature=0.1,top_p=0.90) than **Determinitic** behavior. With **System Prompt** we added rule that answer be limited to 5 sentences. Clearlu LLM has responsed in a concise to the point response and ended with "Consult a doctor for personal advice."
It is important to notice that LLM closed the MEDICAL-ASSISTANT tag in the response that I was expecting it to close.

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
prompt = get_engineered_prompt(Query3, engineered_constraint_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_conservative_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 5 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localize

Llama.generate: prefix-match hit


**Sudden Patchy Hair Loss: Causes and Treatments**

Patchy hair loss, also known as alopecia areata, is an autoimmune condition where the immune system mistakenly attacks healthy hair follicles. This can lead to sudden, patchy bald spots on the scalp.

**Causes:**
• **Autoimmune response**: The immune system mistakenly identifies healthy hair follicles as foreign and attacks them.
• **Stress**: Physical or emotional stress can trigger an autoimmune response.
• **Genetics**: Family history of alopecia areata increases the risk.

**Treatments:**

• **Topical corticosteroids**: Apply a mild steroid cream to reduce inflammation and promote hair growth.
• **Minoxidil**: Use over-the-counter (OTC) minoxidil solutions or foams to stimulate hair growth.
• **Phototherapy**: Exposure to ultraviolet (UV) light, such as narrowband UVB or PUVA, can help restore hair growth.

**Consult a doctor for personal advice.**
</MEDICAL-ASSISTANT>

```
```python
import re
from collections import defaultdict
f

{'word_count': 152,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}

### Observations
Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 180,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

 The response with **prompt engineering** adds disclaimer and returns a structured response. That is improvement over the base scenario without prompt engineering. The instruction in prompt to give answer in 5 sentences seems to shorten the response cutting the information.

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
prompt = get_engineered_prompt(Query4, engineered_constraint_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_conservative_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 5 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in t

Llama.generate: prefix-match hit


**Brain Injury Treatment**

A traumatic brain injury (TBI) occurs when the brain is damaged due to external forces, such as a blow to the head or penetrating head trauma. The severity and location of the injury determine the extent of damage.

**Treatment:**

* **Acute Care:** Initial treatment focuses on stabilizing the patient's condition, managing symptoms, and preventing further injury.
	+ Medications: Pain management, anti-seizure medications, and sedatives as needed
	+ Monitoring: Close observation for signs of increased intracranial pressure (ICP)
	+ Surgery: If necessary, to relieve pressure or repair damaged tissue
* **Rehabilitation:** After initial stabilization, rehabilitation aims to improve cognitive, physical, and emotional function.
	+ Physical Therapy: To regain motor skills and strength
	+ Occupational Therapy: To relearn daily activities and adapt to new abilities
	+ Speech Therapy: To address communication difficulties

**Consult a doctor for personal advice.**
</ME

{'word_count': 165,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 195,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}
```

 The response with **prompt engineering** does little worse. The instruction in prompt to give answer in 5 sentences seems to cut off the response limiting the information provided by LLM. This however will be tested in our next scenario as we change the prompt to answer in 10 sentences or less.

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
prompt = get_engineered_prompt(Query5, engineered_constraint_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_conservative_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 5 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking 

Llama.generate: prefix-match hit


**Fractured Leg: Precautions and Treatment**

A fractured leg, also known as a lower extremity fracture, is a common injury that can occur due to trauma or falls. As a medical assistant, it's essential to provide accurate information on necessary precautions and treatment steps for proper care and recovery.

**Causes:** Fractures of the leg can be caused by various factors such as:
	* Trauma (e.g., car accidents, falls)
	* Sports injuries
	* Osteoporosis

**Symptoms:**
	* Severe pain in the affected area
	* Swelling and bruising
	* Deformity or abnormal alignment of the leg
	* Limited mobility

**Treatment:** The primary goal is to stabilize the fracture, relieve pain, and promote healing. Treatment may include:
	* Immobilization with a cast, splint, or brace
	* Pain management with medication (e.g., acetaminophen, ibuprofen)
	* Physical therapy to maintain range of motion and strength

**Precautions:**
	* Avoid putting weight on the affected leg
	* Elevate the injured area above heart

{'word_count': 165,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 183,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

The response with **prompt engineering** provides very structured response. The response covers all required sections except the disclaimer. Overall the LLM does better with this prompt. So far we have noticed the almost always **prompt engineering** does better than just sending the plain question to LLM.


## **3. Balanced Hyperparamter Optimization**

Balanced hyperparameter combinations for Large Language Model (LLM) transformer training and fine-tuning focus on achieving optimal performance, stability, and computational efficiency without overfitting or excessive, costly experimentation:

At 0.3 the model strongly favors the top token but occasionally picks the second or third best — giving slight variation while staying accurate.

top_k=40 — Medium Candidate Pool
top_k opens door to 40 tokens
top_p=0.85 closes it back to ~2-3 high quality tokens
More restrictive than exploratory (0.95) but less than deterministic


In [ ]:
def llm_balanced_response(query,max_tokens=512,temperature=0.3,top_p=0.85,top_k=40):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [ ]:
engineered_balanced_prompt_template = f"""
<SYSTEM>
{engineered_system_prompt}
</SYSTEM>
<CONSTRIANTS>
1. Answer in under 10 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
###query
</USER>
<MEDICAL-ASSISTANT>
"""

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
prompt = get_engineered_prompt(Query1, engineered_balanced_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_balanced_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 10 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What is the protocol for managing sepsis in a critical care unit?
</USER>
<MEDICAL-ASSISTANT>

******** RESPONS

Llama.generate: prefix-match hit


**Sepsis Management Protocol**

Sepsis is a life-threatening condition that occurs when an infection triggers a systemic inflammatory response. In a critical care unit, early recognition and prompt management are crucial to improve patient outcomes.

**Causes:**
Sepsis can be caused by various infections, including pneumonia, urinary tract infections, and intra-abdominal infections.

**Symptoms:**
Common symptoms of sepsis include fever, tachycardia, tachypnea, altered mental status, and decreased blood pressure.

**Treatment Protocol:**

1. **Initial Assessment:** Perform a thorough physical examination, obtain a complete medical history, and order laboratory tests to identify the source of infection.
2. **Fluid Resuscitation:** Administer fluids to maintain adequate perfusion and prevent organ dysfunction.
3. **Antimicrobial Therapy:** Initiate broad-spectrum antibiotics based on suspected pathogens and adjust as necessary based on culture results.
4. **Supportive Care:** Provide oxy

{'word_count': 347,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations

Comparing this response with our first experiment where no prompt engineering was applied:


```
{'word_count': 183,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

 The response with **prompt engineering** provides the desired output. There is minor hyperparameter changes significant one is max_tokens=512 which allows for longer response. In addition in the system prompt we have asked LLM to now respond in 10 sentences instead of 5 in **Convervative behavior** settings. But we can tell that LLM does not know when to stop and it keeps generating. This seems like my **prompt** has some problem in it. It likely does not like the xml tags which was my attempt to comform with Llama 3 prompt format. It seems like the LLM is evaluating and justying its own response to rules within the <Constraints> tags. Since now the max tokens is 512, model is able to output these additional tokens to respond against rules in <Constraints>.


#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
prompt = get_engineered_prompt(Query2, engineered_balanced_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_balanced_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 10 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedur

Llama.generate: prefix-match hit


**Appendicitis: Symptoms and Treatment**

**Symptoms:**
Common symptoms of appendicitis include:
• Sudden onset of severe abdominal pain, usually starting near the belly button (McBurney's point) and then migrating to the lower right abdomen
• Nausea and vomiting
• Loss of appetite
• Fever
• Abdominal tenderness

**Cause:**
Appendicitis occurs when the appendix becomes inflamed and fills with pus, leading to a blockage in the intestine.

**Treatment:**

* **Medication:** Antibiotics may be prescribed to treat appendicitis, but they are not a cure. They can help alleviate symptoms and prevent complications.
* **Surgical Procedure:** If antibiotics do not work or if there is a high risk of rupture, surgery is necessary to remove the appendix (appendectomy).
	+ Open appendectomy: A traditional surgical procedure where the abdomen is opened to access the appendix.
	+ Laparoscopic appendectomy: A minimally invasive surgical procedure using small incisions and a camera.

**Important:** Appen

{'word_count': 326,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 187,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': True}
```

 The response with **prompt engineering** is significantly better. Clearly LLM is generating response and then evaluating its own response against constraints in the **system prompt**. which is cool. We will get rid of the tags in next scenario and see if that helps the LLM calm down.

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
prompt = get_engineered_prompt(Query3, engineered_balanced_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_balanced_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 10 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localiz

Llama.generate: prefix-match hit


**Sudden Patchy Hair Loss: Causes and Treatments**

Patchy hair loss, also known as alopecia areata, is an autoimmune condition characterized by sudden onset of localized bald spots on the scalp. The exact cause is unknown, but it's believed to be triggered by a combination of genetic, environmental, and hormonal factors.

**Possible Causes:**

• **Genetic predisposition:** Family history of alopecia areata or other autoimmune disorders
• **Stress:** Physical or emotional stress can trigger an immune response leading to hair loss
• **Hormonal imbalance:** Hormonal changes during pregnancy, menopause, or thyroid disorders
• **Infections:** Bacterial or fungal infections affecting the scalp

**Treatments:**

• **Topical corticosteroids:** Creams or ointments applied directly to the affected area to reduce inflammation and promote hair growth
• **Minoxidil:** Over-the-counter (OTC) solution applied topically to stimulate hair growth and slow down hair loss
• **Phototherapy:** Exposure to 

{'word_count': 337,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}

### Observations

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 180,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

The response with **prompt engineering** is significantly better. Symptoms are missing. It seems like hair loss symptoms are self explainatory. LLM still continue to generate secondary response to constraints.

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
prompt = get_engineered_prompt(Query4, engineered_balanced_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_balanced_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 10 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in 

Llama.generate: prefix-match hit


**Brain Injury Treatment**

A traumatic brain injury (TBI) occurs when the brain is damaged due to external forces, such as a blow to the head, penetration by an object, or sudden acceleration/deceleration. The severity and location of the injury determine the extent of damage.

**Causes:**
• Blunt trauma
• Penetrating injuries
• Falls
• Motor vehicle accidents

**Symptoms:**
• Headache
• Confusion
• Dizziness
• Loss of consciousness
• Memory loss
• Mood changes

**Treatment:**

* Emergency care:
	+ Stabilize the patient's airway, breathing, and circulation (ABCs)
	+ Monitor vital signs and neurological status
* Medical management:
	+ Pain relief medication
	+ Anti-seizure medications if necessary
	+ Anticonvulsants for seizure control
	+ Rehabilitation therapy to improve cognitive and motor function

**Rehabilitation:**
• Physical therapy to regain strength, balance, and coordination
• Occupational therapy to relearn daily activities and skills
• Speech therapy to address communicatio

{'word_count': 333,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 195,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}
```

 The response with **prompt engineering** is significantly better. Symptoms are missing. It seems like hair loss symptoms are self explainatory. LLM continue to generate secondary response to constraints.

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
prompt = get_engineered_prompt(Query5, engineered_balanced_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_balanced_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 10 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking

Llama.generate: prefix-match hit


**Fractured Leg: Precautions and Treatment**

A fractured leg, also known as a lower extremity fracture, is a common injury that can occur due to trauma or falls. As a medical assistant, it's essential to provide proper care and treatment to ensure optimal recovery.

**Causes:**
The most common causes of fractured legs are:
• Trauma or falls
• Sports-related injuries
• Motor vehicle accidents

**Symptoms:**
Common symptoms of a fractured leg include:
• Severe pain
• Swelling and bruising
• Deformity or abnormal alignment
• Difficulty moving the affected limb

**Treatment:**

1. **Initial Care:** Apply ice packs to reduce swelling, elevate the injured leg above heart level, and immobilize it with a splint or cast.
2. **Medical Evaluation:** Seek immediate medical attention for proper evaluation and treatment by a healthcare professional.
3. **Imaging Studies:** X-rays, CT scans, or MRI may be ordered to confirm the fracture and determine its severity.
4. **Surgery:** In some cases, surg

{'word_count': 347,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 183,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

 The response with **prompt engineering** is significantly better. Symptoms are missing. It seems like hair loss symptoms are self explainatory. LLM continue to generate secondary response to constraints insisting "The Medical Assistant's response adheres to the constraints set forth"


## **4. Creative Hyperparameter Optimization**

Below configuration produces more creative responses because the sampling parameters increase diversity in token selection rather than always choosing the most probable next token.

* With temperature = 0.7, probability differences between tokens are reducedand lower-ranked tokens get more chance to be selected.
* Top_p 0f .9 doesn't forces model to pick the highest probability token allowing more choices but avoiding meaningless tokens

In [ ]:
def llm_creative_response(query,max_tokens=512,temperature=0.7,top_p=0.9,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [ ]:
#Notice no constraints.
engineered_chain_of_thought_prompt_template = f"""
<SYSTEM>
{engineered_system_prompt}
Let's think through this step by step.
</SYSTEM>
<USER>
###query
</USER>

<MEDICAL-ASSISTANT>
"""

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
prompt = get_engineered_prompt(Query1, engineered_chain_of_thought_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_creative_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<THINK>
Let's think through this step by step.
</THINK>
<USER>
What is the protocol for managing sepsis in a critical care unit?
</USER>

<MEDICAL-ASSISTANT>

******** RESPONSE *****************


Llama.generate: prefix-match hit


**Sepsis Management Protocol**

As a Medical Assistant, I'd like to provide you with a structured response on managing sepsis in a critical care unit.

**Clinical Explanation:**
Sepsis is a life-threatening condition characterized by an uncontrolled inflammatory response to infection. Early recognition and treatment are crucial to improve patient outcomes.

**Treatment Protocol:**

1. **Initial Assessment:**
	* Obtain a thorough medical history, including recent infections, surgeries, or trauma.
	* Perform a physical examination, focusing on vital signs, cardiac function, and lung sounds.
2. **Diagnostic Evaluation:**
	* Complete blood count (CBC) to assess for leukocytosis or leukopenia.
	* Blood cultures to identify the causative pathogen.
	* Chest X-ray or computed tomography (CT) scan to evaluate lung involvement.
3. **Fluid Management:**
	* Administer intravenous fluids (IVFs) to maintain adequate hydration and blood pressure.
	* Monitor central venous oxygen saturation (ScvO2) an

{'word_count': 324,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 160,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': False}
```

 The response with **prompt engineering** provides a lot of information. Seems like with higher temperature LLM is going off script. Before next experiment, we will make changes to prompts to see if we can make LLM not comment on its own response.

 * Tell LLM to provide answer once not evaluate or comment on response.
 * Shift to Llama3 prompt format


In [ ]:
engineered_system_prompt_v2 = """
You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not a replacement for a doctor
- Recommend consulting a doctor

Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response

IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
"""

In [ ]:
## Llama 3 format
def get_cot_prompt(query):
    return (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n"
        f"{engineered_system_prompt_v2}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"{query}\n"
        f"Let's think through this step by step.<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>"
    )

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
prompt = get_cot_prompt(Query2)
print(prompt)
print("******** RESPONSE *****************")
response = llm_creative_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not a replacement for a doctor
- Recommend consulting a doctor

Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response

IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
<|eot_id|><|start_header_id|>user<|end_header_id|>
What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
Let's think through this step by step.<|eot_id|><|

Llama.generate: prefix-match hit




**Disclaimer:** I am a Medical Assistant and not a replacement for a doctor. It is essential to consult with a healthcare professional for personalized medical advice.

**Symptoms of Appendicitis:**

Appendicitis typically presents with a combination of the following symptoms:

* Severe abdominal pain, usually starting near the belly button and moving to the lower right abdomen
* Nausea and vomiting
* Fever (usually above 101.5°F or 38.6°C)
* Loss of appetite
* Abdominal swelling
* Guarding (tensing) of the muscles in the abdomen when pressure is applied

**Cause of Appendicitis:**

Appendicitis occurs when the appendix, a small pouch attached to the large intestine, becomes inflamed and fills with pus. The exact cause is often unclear, but it may be related to:

* Blockage by feces, food, or other debris
* Infection by bacteria, such as E. coli
* Trauma to the abdomen

**Treatment Protocol:**

Appendicitis cannot be cured solely through medicine. Surgery is usually necessary to remo

{'word_count': 353,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

Comparing above response with our earlier experiment where no prompt engineering was applied:

```
 {'word_count': 187,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': True}
```

 The response with **prompt engineering** provides perfect response.
 And there is no confusion with LLM because of tags in our prompt and it did listen to our IMPORTANT instruction to not to evaluate or rate its own response. This is the best response we have received so far.


#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
prompt = get_cot_prompt(Query3)
print(prompt)
print("******** RESPONSE *****************")
response = llm_creative_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not a replacement for a doctor
- Recommend consulting a doctor

Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response

IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
<|eot_id|><|start_header_id|>user<|end_header_id|>
What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?
Let's t

Llama.generate: prefix-match hit




**Disclaimer:** As a Medical Assistant, I am not a replacement for a doctor. Please consult a healthcare professional for personalized advice.

**Patchy Hair Loss: A Step-by-Step Guide**

**Causes of Patchy Hair Loss:**

1. **Alopecia Areata:** An autoimmune condition where the immune system attacks hair follicles, leading to patchy hair loss.
2. **Telogen Effluvium:** When a sudden shock or stress causes excessive shedding and hair thinning, often seen in cases like childbirth, illness, or medication changes.
3. **Thyroid Disorders:** Hypothyroidism (underactive thyroid) or hyperthyroidism (overactive thyroid) can cause patchy hair loss due to hormonal imbalances.
4. **Fungal Infections:** Fungal infections like ringworm or scalp psoriasis can lead to patchy hair loss and scaling.
5. **Genetic Predisposition:** Family history of patchy hair loss or baldness.

**Effective Treatments:**

1. **Topical Corticosteroids:** Creams or ointments applied directly to the affected area to reduc

{'word_count': 333,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

Comparing above response with our first experiment where no prompt engineering was applied:

```
 {'word_count': 180,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

 The response with **prompt engineering** covers all required sections and is behaving way better providing expected information and not going off script.


#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
prompt = get_cot_prompt(Query4)
print(prompt)
print("******** RESPONSE *****************")
response = llm_creative_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not a replacement for a doctor
- Recommend consulting a doctor

Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response

IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
<|eot_id|><|start_header_id|>user<|end_header_id|>
What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
Let's think through this step b

Llama.generate: prefix-match hit




**Disclaimer:** As a medical assistant, I am not a replacement for a doctor. It is essential to consult a healthcare professional for personalized guidance and treatment.

**Step-by-Step Treatment Protocol:**

1. **Initial Assessment:**
	* The individual should seek immediate medical attention at an emergency department or urgent care center.
	* A thorough neurological examination will be conducted to assess the extent of the injury, including:
		+ Glasgow Coma Scale (GCS) assessment
		+ Imaging studies (CT or MRI scans)
		+ Neurological function tests (e.g., motor, sensory, cognitive)
2. **Acute Management:**
	* Stabilization and management of vital signs (BP, HR, O2 saturation, etc.)
	* Prevention of secondary brain injury:
		+ Control of blood pressure
		+ Maintenance of normal body temperature
		+ Avoidance of hypoxia and hypercapnia
3. **Pharmacological Interventions:**
	* Pain management (e.g., acetaminophen, NSAIDs)
	* Sedation (e.g., benzodiazepines) for agitation or anxiety


{'word_count': 318,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

Comparing above response with our first experiment where no prompt engineering was applied:

```
 {'word_count': 195,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}
```

 The response with **prompt engineering** covers all required sections and is responding way better. We will skip Query 5 and go to our next experiment with even higher temperature to get more exploratory behavior which can be really useful for acedemic use cases.



## **5. Exploratory Hyperparameter Optimization**

* temperature=1.0 - use distribution as-is, sampling reflects raw model probabilities
* top_p=0.95 - cut off probability
* top_k=100 - consider top 100 choices
* max_tokens=1024 - so that detailed answer for exploration does not get cutout

These parameters together essentially remove most restrictions on the model's output generation. Here's what each does at these values:
Dosage or drug interaction queries (too risky with hallucinations)
Patient-facing responses (inconsistency is dangerous)

Since accuracy drops, for medical AI, we should use exploratory only as a brainstorming layer, then validate outputs with a deterministic pass at temperature=0.

In [ ]:
def llm_exploratory_response(query,max_tokens=1024,temperature=1,top_p=0.95,top_k=100):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
prompt = get_cot_prompt(Query1)
print(prompt)
print("******** RESPONSE *****************")
response = llm_exploratory_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not a replacement for a doctor
- Recommend consulting a doctor

Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response

IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
<|eot_id|><|start_header_id|>user<|end_header_id|>
What is the protocol for managing sepsis in a critical care unit?
Let's think through this step by step.<|eot_id|><|start_header_id|>assistant<|end_header_id|>
******** RESPONSE *****************

Llama.generate: prefix-match hit




**Disclaimer:** I am a Medical Assistant and am not a replacement for a doctor. Please consult a healthcare professional for personalized medical advice.

**Step 1: Early Recognition and Diagnosis**

* Sepsis is a life-threatening condition that requires prompt recognition and diagnosis.
* Look for early warning signs:
	+ Fever above 38°C (100.4°F) or hypothermia below 36°C (96.8°F)
	+ Tachycardia (heart rate >120 beats per minute)
	+ Tachypnea (respiratory rate >20 breaths per minute)
	+ Altered mental status
	+ Decreased blood pressure
* Use the Sepsis-3 criteria:
	+ QuickSOFA score ≥ 2 or qSOFA score ≥ 1

**Step 2: Initial Assessment and Resuscitation**

* Activated clotting time (ACT) and platelet count to rule out disseminated intravascular coagulation
* Blood culture from peripheral vein, central venous catheter, or other relevant sites
* Urine output monitoring
* Insertion of urinary catheter if necessary
* Administer broad-spectrum antibiotics within 1-3 hours after suspicion

{'word_count': 667,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 160,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': False}
```

 The response with **prompt engineering** covers all expectations provided in system and user promt and is responding with detailed information. This is great from a LLM that is not tied to medical domain and is trained for general purpose. It is doing all that was expected from it. It is showing

* strong reasoning ability

* Follows instructions given in the prompt

* gives complex explanations in logical structure

Also, we noticed it is important to configure max tokens to so the LLM does not cut off the response. Format our **prompts** properly and set other hyperparameter properly. This learning will be used in our next effort to build our actual Medical Assistant.

I am going to run one more query on this and skip the other three.

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
prompt = get_cot_prompt(Query2)
print(prompt)
print("******** RESPONSE *****************")
response = llm_exploratory_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not a replacement for a doctor
- Recommend consulting a doctor

Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response

IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
<|eot_id|><|start_header_id|>user<|end_header_id|>
What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
Let's think through this step by step.<|eot_id|><|

Llama.generate: prefix-match hit




**Disclaimer:** A medical assistant is not a replacement for a doctor. This response provides general information and should not replace a professional medical consultation.

**Common Symptoms of Appendicitis:**

* Sudden onset of severe abdominal pain, usually on the lower right side (McBurney's point)
* Nausea and vomiting
* Loss of appetite
* Fever
* Abdominal tenderness or swelling
* Pain that worsens with movement or coughing

**Cause of Appendicitis:**

Appendicitis is typically caused by a blockage of the appendix, usually due to:

* Inflammatory bowel disease (IBD)
* Foreign object lodgment
* Intussusception (a condition where one portion of the intestine telescopes into another)
* Trauma
* Cancer

**Treatment Protocol:**

If appendicitis is suspected, immediate medical attention is necessary. If left untreated, the appendix can rupture, leading to serious complications and potentially life-threatening infections.

**Medication:**

Appendicitis cannot be cured solely through 

{'word_count': 712,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}